# Load and initialise

In [ ]:
import torch
import torch.nn as nn
from models import GenTeacher, GenClassifier, classifier_layels_list
from utils import save_model
from utils.dataset import get_datasets, image_transform
from itertools import chain

device = 'cuda:0'
teacher_model = GenTeacher('ViT-B/32', device)
tea_classifier_model = GenClassifier(512, 100, classifier_layels_list, device)
train_loader, test_loader = get_datasets(image_transform, 500, 'cifar100')

# Pre-trained classification networks

In [ ]:
teacher_model.eval()
tea_classifier_model.train()
optimizer = torch.optim.Adam(tea_classifier_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
ACC_CLA = []
LOSS_CLA = []
for epoch in range(5):
    for i, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        feature = teacher_model(data)
        output = tea_classifier_model(feature)

        loss = criterion(output, target)
        LOSS_CLA.append(loss.item())
        acc = (output.argmax(1) == target).float().mean()
        ACC_CLA.append(acc.item())

        loss.backward()
        optimizer.step()

        if i%10 == 0:
            print(f'Epoch: {epoch}, Loss: {loss.item():.4f}, Acc: {acc.item():.4f}')

# Co-train ViT and classification networks

In [ ]:
teacher_model.train()
tea_classifier_model.train()
optimizer = torch.optim.Adam(chain(teacher_model.parameters(), tea_classifier_model.parameters()), lr=1e-4)
criterion = nn.CrossEntropyLoss()
ACC_JOINT = []
LOSS_JOINT = []
for epoch in range(5):
    for i, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        feature = teacher_model(data)
        output = tea_classifier_model(feature)

        loss = criterion(output, target)
        LOSS_JOINT.append(loss.item())
        acc = (output.argmax(1) == target).float().mean()
        ACC_JOINT.append(acc.item())

        loss.backward()
        optimizer.step()

        if i%10 == 0:
            print(f'Epoch: {epoch}, Loss: {loss.item():.4f}, Acc: {acc.item():.4f}')

# Save model weights

In [ ]:
save_model(teacher_model, 'results/weights/retrain_teacher/teacher_retrain.pt')
save_model(tea_classifier_model, 'results/weights/retrain_teacher/teacher_classifier_retrain.pt')